In [ ]:
import random
from collections import defaultdict, Counter

D = """
the cat sat on the mat
the cat sat on the rug
the dog sat on the mat
the dog ran to the park
the cat ran to the park
the dog sat on the rug
"""
tokens = ["", ""] + list(set(D.split()))
print(f"Vocabulary: {tokens}\n")

# 1. Take a list of tokenized sentences as training data
# We split by line, then by word, and wrap each sentence with  and
raw_sentences = [line.strip().split() for line in D.strip().split('\n') if line.strip()]
training_data = [[""] + sentence + [""] for sentence in raw_sentences]

# 2. Count transitions between consecutive tokens
# We'll use a nested dictionary where transitions[current][next] = count
transitions = defaultdict(Counter)

for sentence in training_data:
    # Iterate through pairs of consecutive tokens
    for i in range(len(sentence) - 1):
        current_token = sentence[i]
        next_token = sentence[i + 1]
        transitions[current_token][next_token] += 1

# 3. Construct the conditional distribution P(Xt | Xt-1)
probabilities = defaultdict(dict)

for current_token, next_tokens in transitions.items():
    total_transitions = sum(next_tokens.values())
    for next_token, count in next_tokens.items():
        # P(Next | Current) = Count(Current -> Next) / Total transitions from Current
        probabilities[current_token][next_token] = count / total_transitions

# 4. Display the probabilities for a specified previous token
def show_probabilities(prev_token):
    print(f"P(X_t | X_t-1 = '{prev_token}'):")
    if prev_token not in probabilities:
        print("  Token not found in history.")
        return

    for next_token, prob in probabilities[prev_token].items():
        print(f"  {next_token}: {prob:.4f}")

# 5. Predict the most probable next token (Greedy / Argmax)
def predict_most_probable(prev_token):
    if prev_token not in probabilities:
        return None
    # Find the token with the highest probability value
    best_token = max(probabilities[prev_token].items(), key=lambda x: x[1])[0]
    return best_token

# 6 & 7. Generate a sentence by repeatedly sampling, stopping at
import random

def generate_sentence(mode="sampling", max_tokens=25):
    """
    mode='greedy': Mode A - Always choose argmax P(w | w_prev)
    mode='sampling': Mode B - Sample from P(w | w_prev)
    """
    current_token = "<START>"
    generated_tokens = []
    
    for _ in range(max_tokens):
        if current_token not in probabilities or not probabilities[current_token]:
            break
            
        next_choices = list(probabilities[current_token].keys())
        weights = list(probabilities[current_token].values())
        
        if mode == "greedy":
            # Mode A: Choose token with the maximum conditional probability
            next_token = max(probabilities[current_token].items(), key=lambda x: x[1])[0]
        elif mode == "sampling":
            # Mode B: Sample proportionally from the distribution
            next_token = random.choices(next_choices, weights=weights, k=1)[0]
        else:
            raise ValueError("Mode must be 'greedy' or 'sampling'")
            
        if next_token == "<END>":
            break
            
        generated_tokens.append(next_token)
        current_token = next_token
        
    return " ".join(generated_tokens)

# --- Generate 5 sentences using each method ---
print("--- Mode A: Greedy Generation ---")
for i in range(5):
    print(f"{i+1}. {generate_sentence(mode='greedy')}")

print("\n--- Mode B: Probabilistic Sampling ---")
for i in range(5):
    print(f"{i+1}. {generate_sentence(mode='sampling')}")


# --- Demonstrating the functionality ---

# Show distribution for a specific token
show_probabilities("the")
print()

show_probabilities("cat")
print()

# Predict highest probability next token
prev = "sat"
best_next = predict_most_probable(prev)
print(f"Most probable token after '{prev}': '{best_next}'\n")

# Generate a few random sentences
print("Generated sentences:")
for i in range(5):
    print(f"{i+1}. {generate_sentence()}")

: 